In [ ]:
%cd /content/drive/MyDrive/Mesh-Network-Analysis-Main-Library

In [ ]:
!top -b -n 1 | grep python

In [ ]:
import os
import gzip
import sqlite3
import time
import shutil
import tempfile
import hashlib
import multiprocessing
import xml.etree.ElementTree as ET
from pathlib import Path
from uuid import uuid4

# ==========================================
# 1. ENVIRONMENT CONFIGURATION
# ==========================================
# DESTINATION_DIR: The final storage location (Google Drive, AWS EFS, Local HDD)
DESTINATION_DIR = Path("/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/db_test")
BASELINE_DIR = Path("/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/pubmed_baseline")

DESTINATION_DIR.mkdir(parents=True, exist_ok=True)
DESTINATION_DB_PATH = DESTINATION_DIR / "master_mesh_database.db"

# LOCAL_WORKSPACE: The physical VM NVMe drive for maximum IOPS
LOCAL_WORKSPACE = Path(tempfile.gettempdir()) / "mesh_etl_workspace"
LOCAL_WORKSPACE.mkdir(exist_ok=True)
LOCAL_DB_PATH = LOCAL_WORKSPACE / "local_active_master.db"
LOCAL_XML_DIR = LOCAL_WORKSPACE / "xml_buffer"
LOCAL_XML_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 25
CHECKPOINT_INTERVAL = 4  # Execute cryptographic backup every 4 chunks (100 files)

# ==========================================
# 2. CRYPTOGRAPHIC VERIFICATION ENGINE
# ==========================================
def get_file_hash(filepath):
    """Calculates the SHA-256 digital fingerprint of a file in low-memory blocks."""
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

def verified_safe_transfer(src_path, dest_path, max_retries=5):
    """Copies a file and strictly verifies the copy is identical byte-for-byte."""
    print(f"      [Sync] Profiling local database fingerprint...", end=" ", flush=True)
    src_size = os.path.getsize(src_path)
    src_hash = get_file_hash(src_path)
    print("Done.")

    tmp_dest = Path(str(dest_path) + ".tmp")

    for attempt in range(1, max_retries + 1):
        print(f"      [Sync] Attempt {attempt}/{max_retries} - Transferring to storage...", end=" ", flush=True)
        try:
            if tmp_dest.exists(): tmp_dest.unlink()

            shutil.copy2(src_path, tmp_dest)
            os.sync()

            dest_size = os.path.getsize(tmp_dest)
            if dest_size != src_size:
                print(f"FAIL (Size mismatch: {dest_size} != {src_size})")
                continue

            dest_hash = get_file_hash(tmp_dest)
            if src_hash == dest_hash:
                os.replace(tmp_dest, dest_path)
                print("SUCCESS (Cryptographic Match Verified).")
                return True
            else:
                print(f"FAIL (Hash mismatch detected).")
                continue

        except Exception as e:
            print(f"ERROR ({e})")
            time.sleep(5)

    raise Exception("\n[CRITICAL ERROR] Failed to transfer a healthy database after maximum retries.")

# ==========================================
# 3. MAP-REDUCE SHARD WORKER
# ==========================================
def build_local_shard(local_file_chunk):
    """Executes on parallel CPU cores to parse XML chunks into local SQLite shards."""
    shard_path = LOCAL_WORKSPACE / f"shard_{uuid4().hex}.db"
    conn = sqlite3.connect(shard_path)

    conn.execute("PRAGMA journal_mode = OFF;")
    conn.execute("PRAGMA synchronous = OFF;")
    conn.execute("CREATE TABLE shard_data (pmid INTEGER, mesh_terms TEXT, source_file TEXT)")

    processed_names = []
    total_articles = 0
    cursor = conn.cursor()

    for filepath in local_file_chunk:
        batch = []
        source_filename = Path(filepath).name
        try:
            with gzip.open(filepath, 'rb') as f:
                context = ET.iterparse(f, events=('end',))
                for event, elem in context:
                    if elem.tag == 'PubmedArticle':
                        pmid_node = elem.find('.//PMID')
                        mesh_list = elem.find('.//MeshHeadingList')

                        if pmid_node is not None and mesh_list is not None:
                            pmid = int(pmid_node.text)
                            terms = [f"{'*' if d.get('MajorTopicYN') == 'Y' else ''}{d.text}"
                                     for d in mesh_list.findall('.//DescriptorName')]
                            if terms:
                                batch.append((pmid, ";".join(terms), source_filename))
                        elem.clear()

            if batch:
                cursor.executemany("INSERT INTO shard_data (pmid, mesh_terms, source_file) VALUES (?, ?, ?)", batch)
                total_articles += len(batch)

            processed_names.append(source_filename)
        except Exception as e:
            print(f"\n  [!] Shard Error parsing {source_filename}: {e}")

    conn.commit()
    conn.close()
    return str(shard_path), processed_names, total_articles

# ==========================================
# 4. MASTER ORCHESTRATOR
# ==========================================
def run_environment_agnostic_etl():
    print("<<< Phase 0: Environment Diagnostics & Bootstrapping >>>")

    if DESTINATION_DB_PATH.exists():
        print("  Existing target database detected. Syncing to Local Workspace...")
        verified_safe_transfer(DESTINATION_DB_PATH, LOCAL_DB_PATH)
    else:
        print("  No target database found. Initializing blank Local Workspace.")

    # Initialize Workspace DB
    conn = sqlite3.connect(LOCAL_DB_PATH)
    conn.execute("PRAGMA journal_mode = WAL;")
    conn.execute("CREATE TABLE IF NOT EXISTS master_mesh_annotations (pmid INTEGER, mesh_terms TEXT, source_file TEXT)")
    conn.execute("CREATE TABLE IF NOT EXISTS parsed_files (filename TEXT PRIMARY KEY)")

    cursor = conn.cursor()
    cursor.execute("SELECT filename FROM parsed_files")
    completed_files = {row[0] for row in cursor.fetchall()}

    all_files = list(BASELINE_DIR.glob("*.xml.gz"))
    pending_files = [f for f in all_files if f.name not in completed_files]

    if not pending_files:
        print("  All files parsed! Proceeding to Indexing Phase...")
    else:
        print(f"  Found {len(all_files)} total files. {len(completed_files)} already complete.")
        print(f"  Resuming with {len(pending_files)} files...\n")

        chunks = [pending_files[i:i + CHUNK_SIZE] for i in range(0, len(pending_files), CHUNK_SIZE)]
        cores = max(1, multiprocessing.cpu_count() - 1)
        print(f"<<< Phase 1 & 2: Distributed Parsing & Aggregation ({cores} Cores) >>>")

        start_time = time.time()
        global_articles = 0

        for chunk_idx, chunk in enumerate(chunks, 1):
            local_chunk_paths = []
            for raw_file in chunk:
                local_dest = LOCAL_XML_DIR / raw_file.name
                shutil.copy2(raw_file, local_dest)
                local_chunk_paths.append(local_dest)

            with multiprocessing.Pool(cores) as pool:
                sub_chunks = [local_chunk_paths[i::cores] for i in range(cores)]
                sub_chunks = [sc for sc in sub_chunks if sc]

                for shard_path_str, parsed_names, article_count in pool.imap_unordered(build_local_shard, sub_chunks):
                    cursor.execute(f"ATTACH DATABASE '{shard_path_str}' AS temp_shard")
                    cursor.execute("BEGIN TRANSACTION;")

                    cursor.execute("INSERT INTO master_mesh_annotations (pmid, mesh_terms, source_file) SELECT pmid, mesh_terms, source_file FROM temp_shard.shard_data")
                    for fname in parsed_names:
                        cursor.execute("INSERT INTO parsed_files (filename) VALUES (?)", (fname,))

                    conn.commit()
                    cursor.execute("DETACH DATABASE temp_shard")
                    os.remove(shard_path_str)
                    global_articles += article_count

            for local_file in local_chunk_paths:
                if local_file.exists(): local_file.unlink()

            cursor.execute("SELECT count(*) FROM parsed_files")
            total_done = cursor.fetchone()[0]
            elapsed = time.time() - start_time
            print(f"  -> Processed Block {chunk_idx}/{len(chunks)}. (Total: {total_done}/{len(all_files)}) [+ {global_articles:,} articles] [{elapsed/60:.1f} min]")

            # --- ATOMIC CHECKPOINT ---
            if chunk_idx % CHECKPOINT_INTERVAL == 0:
                conn.commit()
                conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
                conn.execute("PRAGMA journal_mode = DELETE;")
                conn.close()

                verified_safe_transfer(LOCAL_DB_PATH, DESTINATION_DB_PATH)

                conn = sqlite3.connect(LOCAL_DB_PATH)
                conn.execute("PRAGMA journal_mode = WAL;")
                cursor = conn.cursor()

    print("\n<<< Phase 3: Post-Load Optimization & Indexing >>>")
    cursor.execute("SELECT count(*) FROM sqlite_master WHERE type='index' AND name='idx_pmid'")
    if cursor.fetchone()[0] == 0:
        print("  Building B-Tree Index locally...")
        cursor.execute("DROP INDEX IF EXISTS idx_pmid")
        index_start = time.time()
        cursor.execute("CREATE UNIQUE INDEX idx_pmid ON master_mesh_annotations (pmid)")
        print(f"  Index complete. [{((time.time() - index_start)/60):.1f} min]")
    else:
        print("  Index already exists.")

    print("\n<<< Phase 4: Final Network Transfer >>>")
    conn.commit()
    conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
    conn.execute("PRAGMA journal_mode = DELETE;")
    conn.execute("VACUUM;") # Final defragmentation
    conn.close()

    verified_safe_transfer(LOCAL_DB_PATH, DESTINATION_DB_PATH)
    print("\n<<< ETL COMPLETE: Database successfully compiled, verified, and secured. >>>")

if __name__ == '__main__':
    run_environment_agnostic_etl()

In [ ]:
"""
test_mesh_xml_processor.py

STANDALONE TEST SCRIPT:
Extracts MeSH terms, UIs, and Tree Numbers directly from the NLM MeSH XML file.
Bypasses both the deprecated ASCII format and the incomplete MARC format.
"""

import os
import traceback
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd

# <<< Constants >>>
CATEGORIES_TO_KEEP = {'A', 'C', 'D', 'G'}
CHECK_TAGS_TO_STOP = {'Male', 'Female'}

TOP_LEVEL_CATEGORY_NAMES = {
    'A': "Anatomy", 'B': "Organisms", 'C': "Diseases", 'D': "Chemicals and Drugs",
    'E': "Analytical, Diagnostic and Therapeutic Techniques, and Equipment",
    'F': "Psychiatry and Psychology", 'G': "Phenomena and Processes",
    'H': "Disciplines and Occupations",
    'I': "Anthropology, Education, Sociology, and Social Phenomena",
    'J': "Technology, Industry, Agriculture", 'K': "Humanities",
    'L': "Information Science", 'M': "Named Groups", 'N': "Health Care",
    'V': "Publication Characteristics", 'Z': "Geographicals"
}

def extract_all_mesh_data_from_xml(xml_file_path: str, output_csv_path: str, output_py_path: str):
    """
    Reads the MeSH XML file. Extracts Headings, UIs, and Tree Numbers.
    Generates both the annotated CSV and the Python stopwords file.
    """
    print(f"\n" + "<"*30 + ">"*30)
    print(f"<<< Starting Unified XML Extraction >>>")
    print(f"Reading from: {xml_file_path}")
    print("<"*30 + ">"*30 + "\n")

    if not os.path.exists(xml_file_path):
        raise FileNotFoundError(f"XML File Not Found at: {xml_file_path}")

    mesh_terms = []
    mesh_term_to_categories = defaultdict(set)
    all_mh_found = set()

    try:
        # We use iterparse to save RAM (just like the PubMed Baseline parser)
        context = ET.iterparse(xml_file_path, events=('end',))
        count = 0

        for event, elem in context:
            if elem.tag == 'DescriptorRecord':
                count += 1
                if count % 2000 == 0:
                    print(f"  Parsed {count:,} descriptor records...", end='\r')

                # Extract UI and Name
                ui = elem.findtext('.//DescriptorUI')
                name = elem.findtext('.//DescriptorName/String')

                if ui and name:
                    mesh_terms.append([name, ui])
                    all_mh_found.add(name)

                    # Extract Tree Numbers
                    tree_list = elem.find('.//TreeNumberList')
                    if tree_list is not None:
                        for tree_node in tree_list.findall('TreeNumber'):
                            tree_num = tree_node.text
                            if tree_num and tree_num[0].isalpha():
                                top_cat = tree_num[0].upper()
                                mesh_term_to_categories[name].add(top_cat)

                elem.clear() # Clear memory buffer

        print(f"  Parsed {count:,} total records. Analyzing hierarchies...\n")

        # --- PROCESS STOP WORDS ---
        stop_words_grouped = defaultdict(set)
        terms_missing_tns = sorted(list(all_mh_found - set(mesh_term_to_categories.keys())))
        final_orphans_set = set(terms_missing_tns)

        for mh, cats in mesh_term_to_categories.items():
            if mh in CHECK_TAGS_TO_STOP:
                final_orphans_set.add(mh)
                continue

            if not any(c in CATEGORIES_TO_KEEP for c in cats):
                excluded_cats = sorted(list(cats))
                primary_cat = excluded_cats[0] if excluded_cats else 'Unknown'
                stop_words_grouped[primary_cat].add(mh)

        for tag in CHECK_TAGS_TO_STOP:
            if tag in all_mh_found:
                 final_orphans_set.add(tag)

        all_stop_words = final_orphans_set.copy()
        for s in stop_words_grouped.values():
            all_stop_words.update(s)

        write_stopwords_to_python(output_py_path, stop_words_grouped, sorted(list(final_orphans_set)))

        # --- PROCESS DATAFRAME & TAG CSV ---
        print(f"\n<<< Building DataFrame & Tagging Stop Words >>>")
        df = pd.DataFrame(mesh_terms, columns=["DescriptorName", "DescriptorUI"])
        initial_len = len(df)
        df.drop_duplicates(subset=['DescriptorName'], keep='first', inplace=True)

        stop_words_lower = {s.lower() for s in all_stop_words}
        df['MeSH_stop_term'] = df['DescriptorName'].astype(str).str.lower().isin(stop_words_lower)
        stop_count = df['MeSH_stop_term'].sum()

        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        df.to_csv(output_csv_path, index=False)

        print(f"    [+] Successfully extracted {len(df):,} unique terms (from {initial_len:,} raw)")
        print(f"    [+] Tagged {stop_count:,} terms as stop words.")
        print(f"    [+] Saved CSV to '{output_csv_path}'")

    except Exception as e:
        traceback.print_exc()
        raise RuntimeError(f"Unexpected error during XML extraction: {e}")

def write_stopwords_to_python(output_path: str, grouped_stopwords: dict, missing_tns: list):
    """Writes the categorized stop words to a properly formatted Python module."""
    cats_display_list = []
    for cat in sorted(list(CATEGORIES_TO_KEEP)):
        name = TOP_LEVEL_CATEGORY_NAMES.get(cat, "Unknown")
        cats_display_list.append(f"'{cat} - {name}'")
    cats_formatted_str = "[" + ", ".join(cats_display_list) + "]"

    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write('"""\nCombined MeSH Stop Word List\n')
            f.write('Generated by extracting Tree Numbers from MeSH XML file.\n"""\n\n')
            f.write("MESH_STOP_WORDS = [\n")
            f.write("    # This list contains MeSH terms that generally do NOT fall into the primary\n")
            f.write(f"    # MeSH categories: {cats_formatted_str}.\n")
            f.write("    # It includes:\n")
            f.write("    #   1. Terms explicitly categorized outside of the 'kept' categories.\n")
            f.write("    #   2. Terms for which categories could not be determined.\n\n")
            f.write("    # --- Terms Categorized Outside A, C, D, G ---\n")

            sorted_cats = sorted(grouped_stopwords.keys())
            for cat in sorted_cats:
                terms = sorted(list(grouped_stopwords[cat]))
                cat_desc = TOP_LEVEL_CATEGORY_NAMES.get(cat, "Unknown Category")

                f.write("\n")
                f.write("    #---------------------------------------------------------------------------\n")
                f.write(f"    # MeSH Category (source of these stop words): {cat} - {cat_desc}\n")
                f.write("    #---------------------------------------------------------------------------\n")

                for term in terms:
                    clean_term = term.replace('\\', '\\\\').replace('"', '\\"')
                    f.write(f'    "{clean_term}",\n')

            if missing_tns:
                f.write("\n")
                f.write("    #---------------------------------------------------------------------------\n")
                f.write("    # Terms With Undetermined Categories (No valid Tree Numbers found)\n")
                f.write("    #---------------------------------------------------------------------------\n")
                for term in missing_tns:
                    clean_term = term.replace('\\', '\\\\').replace('"', '\\"')
                    f.write(f'    "{clean_term}",\n')

            f.write("]\n")
        print(f"    [+] Successfully generated stop words file: {output_path}")

    except Exception as e:
        raise RuntimeError(f"Error writing Python stop word file: {e}")

if __name__ == "__main__":
    # --- COLAB TEST PATHS ---
    # Update this to wherever you place the 'desc2025.xml' file
    XML_INPUT_FILE = "/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/desc2025.xml"

    TEST_OUTPUT_CSV = "/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/db_test/test_mesh_terms.csv"
    TEST_OUTPUT_PY = "/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/db_test/test_mesh_stopwords.py"

    extract_all_mesh_data_from_xml(XML_INPUT_FILE, TEST_OUTPUT_CSV, TEST_OUTPUT_PY)

In [ ]:
!pip install pymarc

In [ ]:
import os
import requests
import shutil
from pathlib import Path

# Target URL and Destination
XML_URL = "https://nlmpubs.nlm.nih.gov/projects/mesh/2025/xmlmesh/desc2025.xml"
GDRIVE_DESTINATION = Path("/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/raw/desc2025.xml")
LOCAL_TMP_PATH = Path("/content/desc2025.xml")

def download_mesh_xml():
    print(f"Initiating stream download from: {XML_URL}")
    print(f"Target destination: {GDRIVE_DESTINATION}")

    # Ensure destination directory exists
    GDRIVE_DESTINATION.parent.mkdir(parents=True, exist_ok=True)

    # Stream directly to local VM disk first to prevent FUSE timeouts
    try:
        with requests.get(XML_URL, stream=True) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            downloaded_size = 0

            with open(LOCAL_TMP_PATH, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded_size += len(chunk)
                        if total_size > 0:
                            percent = (downloaded_size / total_size) * 100
                            print(f"  Downloading: {percent:.1f}% ({downloaded_size / (1024*1024):.1f} MB)", end='\r')
                        else:
                            print(f"  Downloading: {downloaded_size / (1024*1024):.1f} MB", end='\r')

        print(f"\nDownload complete. Verifying file size...")
        final_size = os.path.getsize(LOCAL_TMP_PATH) / (1024 * 1024)
        print(f"Local file size: {final_size:.2f} MB")

        print("Transferring to Google Drive...")
        shutil.copy2(LOCAL_TMP_PATH, GDRIVE_DESTINATION)

        # Clean up local temporary file
        LOCAL_TMP_PATH.unlink()
        print("XML provisioning sequence successful.")

    except requests.exceptions.RequestException as e:
        print(f"\n[!] Network Error during download: {e}")
    except Exception as e:
        print(f"\n[!] System Error: {e}")

if __name__ == "__main__":
    download_mesh_xml()

In [ ]:
import os
import sqlite3
from pathlib import Path

def run_upsert_test():
    # 1. Setup Isolated Test Environment
    test_dir = Path("data/raw/DB_test")
    test_dir.mkdir(parents=True, exist_ok=True)
    db_path = test_dir / "test_master.db"
    shard_path = test_dir / "temp_shard.db"

    # Clean up previous test runs if they exist
    if db_path.exists(): os.remove(db_path)
    if shard_path.exists(): os.remove(shard_path)

    print(f"<<< Setting up Test Database in {test_dir} >>>")

    # 2. Initialize "Master Database" with the UNIQUE Index
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE master_mesh_annotations (pmid INTEGER, pub_date TEXT, mesh_terms TEXT, source_file TEXT)")
    cursor.execute("CREATE TABLE parsed_files (filename TEXT PRIMARY KEY)")

    # This is the exact index that caused your pipeline to crash previously
    cursor.execute("CREATE UNIQUE INDEX idx_pmid ON master_mesh_annotations (pmid)")

    # 3. Insert Initial "Baseline" Data
    initial_data = [
        (1001, "2020-01-01", "Aspirin", "baseline_01.xml.gz"),
        (1002, "2021-06-15", "Dermatitis", "baseline_01.xml.gz"), # We will test updating this one
        (1003, "2022-11-20", "Liver Cirrhosis", "baseline_01.xml.gz")
    ]
    cursor.executemany("INSERT INTO master_mesh_annotations VALUES (?, ?, ?, ?)", initial_data)
    cursor.execute("INSERT INTO parsed_files VALUES ('baseline_01.xml.gz')")
    conn.commit()

    print("\n--- DATABASE BEFORE DAILY UPDATE ---")
    for row in cursor.execute("SELECT * FROM master_mesh_annotations"):
        print(f"  PMID: {row[0]} | Terms: {row[2]} | File: {row[3]}")

    # 4. Create a "Temp Shard" mimicking a downloaded Daily Update file
    shard_conn = sqlite3.connect(shard_path)
    shard_conn.execute("CREATE TABLE shard_data (pmid INTEGER, pub_date TEXT, mesh_terms TEXT, source_file TEXT)")

    update_data = [
        (1002, "2021-06-15", "Dermatitis;Allergic Contact", "update_01.xml.gz"), # MODIFIED: Curator added 'Allergic Contact'
        (1004, "2024-02-10", "Skin", "update_01.xml.gz")                           # NEW: Brand new published article
    ]
    shard_conn.executemany("INSERT INTO shard_data VALUES (?, ?, ?, ?)", update_data)
    shard_conn.commit()
    shard_conn.close()

    # 5. Perform the UPSERT Logic (The Fix)
    print("\n<<< ATTACHING SHARD AND EXECUTING UPSERT >>>")
    cursor.execute(f"ATTACH DATABASE '{shard_path}' AS temp_shard")

    try:
        # Notice the "OR REPLACE" here - this is what we are changing in baseline_manager.py
        cursor.execute("""
            INSERT OR REPLACE INTO master_mesh_annotations (pmid, pub_date, mesh_terms, source_file)
            SELECT pmid, pub_date, mesh_terms, source_file
            FROM temp_shard.shard_data
        """)

        cursor.execute("INSERT OR REPLACE INTO parsed_files (filename) VALUES ('update_01.xml.gz')")
        conn.commit()
        print("  [SUCCESS] Upsert completed smoothly! No UNIQUE constraint crashes.")
    except Exception as e:
        print(f"  [CRITICAL ERROR] Crash occurred: {e}")

    cursor.execute("DETACH DATABASE temp_shard")

    # 6. Verify Results
    print("\n--- DATABASE AFTER DAILY UPDATE ---")
    for row in cursor.execute("SELECT * FROM master_mesh_annotations"):
        if row[0] == 1002:
            print(f"  PMID: {row[0]} | Terms: {row[2]} | File: {row[3]}  <-- [Successfully Overwritten!]")
        elif row[0] == 1004:
            print(f"  PMID: {row[0]} | Terms: {row[2]} | File: {row[3]}  <-- [Successfully Appended!]")
        else:
            print(f"  PMID: {row[0]} | Terms: {row[2]} | File: {row[3]}  <-- (Untouched)")

    print("\n--- PARSED FILES TRACKER ---")
    for row in cursor.execute("SELECT * FROM parsed_files"):
        print(f"  {row[0]}")

    conn.close()

    # Cleanup temp shard
    if shard_path.exists(): os.remove(shard_path)

if __name__ == "__main__":
    run_upsert_test()

In [ ]:
import networkx as nx
import time
import random

def test_eigenvector_centrality():
    print("<<< Generating Synthetic Test Network >>>")
    # Generate a large network similar to a citation graph (Scale-free)
    # 5,000 nodes and roughly 50,000 edges
    G = nx.barabasi_albert_graph(50000, 10)

    # Add random weights to simulate the co-occurrence weights in your pipeline
    for u, v in G.edges():
        G[u][v]['weight'] = random.uniform(0.1, 1.0)

    print(f"Network created with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.\n")

    # --- Method 1: Standard Python (Strict Tolerance) ---
    print("Test 1: Standard NetworkX (tol=1.0e-6)")
    start_time = time.time()
    ev_standard = nx.eigenvector_centrality(G, weight='weight', max_iter=1000, tol=1.0e-6)
    time_standard = time.time() - start_time
    print(f"  -> Time: {time_standard:.4f} seconds")

    # --- Method 2: Standard Python (Loose Tolerance) ---
    print("\nTest 2: Standard NetworkX (tol=1.0e-4)")
    start_time = time.time()
    ev_loose = nx.eigenvector_centrality(G, weight='weight', max_iter=1000, tol=1.0e-4)
    time_loose = time.time() - start_time
    print(f"  -> Time: {time_loose:.4f} seconds")

    # --- Method 3: NumPy/SciPy Optimized ---
    print("\nTest 3: NumPy/SciPy Optimized (tol=1.0e-4)")
    start_time = time.time()
    try:
        ev_numpy = nx.eigenvector_centrality_numpy(G, weight='weight', max_iter=1000, tol=1.0e-6)
        time_numpy = time.time() - start_time
        print(f"  -> Time: {time_numpy:.4f} seconds")
    except Exception as e:
        print(f"  -> Failed: {e}")
        return

    # --- Validation: Do they produce the exact same relative rankings? ---
    print("\n<<< Validating Results (Top 5 Nodes) >>>")

    # Helper function to get top 5 nodes by score
    def get_top_5(ev_dict):
        return sorted(ev_dict.items(), key=lambda x: x[1], reverse=True)[:5]

    top_standard = get_top_5(ev_standard)
    top_loose = get_top_5(ev_loose)
    top_numpy = get_top_5(ev_numpy)

    print(f"Standard (1e-6) Top 5 Nodes: {[n for n, score in top_standard]}")
    print(f"Standard (1e-4) Top 5 Nodes: {[n for n, score in top_loose]}")
    print(f"NumPy    (1e-4) Top 5 Nodes: {[n for n, score in top_numpy]}")

    print("\n<<< Conclusion >>>")
    if time_numpy > 0:
        print(f"NumPy was {time_standard / time_numpy:.1f}x faster than the standard strict method.")
        print(f"Loosening tolerance without NumPy was {time_standard / time_loose:.1f}x faster.")

if __name__ == "__main__":
    test_eigenvector_centrality()

In [ ]:
%cd /content/drive/MyDrive/Mesh-Network-Analysis-Main-Library

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

# --- 1. MOUNT GOOGLE DRIVE ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

# --- 2. DEFINE PATHS ---
PROCESSED_DIR = "/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/processed"
CLEANED_DB = os.path.join(PROCESSED_DIR, "DAC_Mesh_cleaned_pmids.db")

def run_citation_forensics():
    if not os.path.exists(CLEANED_DB):
        print(f"[!] DATABASE NOT FOUND: {CLEANED_DB}")
        return

    # Connect in STRICTLY READ-ONLY mode to prevent file modification
    db_uri = f"file:{CLEANED_DB}?mode=ro"
    conn = sqlite3.connect(db_uri, uri=True)
    cursor = conn.cursor()

    try:
        # TEST 1: Look up the exact Top 5 PMIDs
        target_pmids = ['26179009', '33093296', '12354389', '18431959', '9927693']
        placeholders = ','.join('?' for _ in target_pmids)

        print("\n--- TEST 1: Checking Target PMIDs ---")
        cursor.execute(f"SELECT pmid, cited_by FROM pmids_table WHERE pmid IN ({placeholders})", target_pmids)
        top_5_rows = cursor.fetchall()

        if not top_5_rows:
            print("[!] Target PMIDs not found in the master database.")
        else:
            for row in top_5_rows:
                print(f"  PMID {row[0]}: Raw cited_by = {repr(row[1])}")

        # TEST 2: Find ANY articles with valid citations
        print("\n--- TEST 2: Scanning Master DB for Valid Citations ---")
        cursor.execute("""
            SELECT pmid, cited_by
            FROM pmids_table
            WHERE cited_by IS NOT NULL
              AND cited_by != ''
              AND cited_by != 'None'
              AND cited_by != 'NaN'
            LIMIT 5
        """)
        valid_rows = cursor.fetchall()

        if not valid_rows:
            print("[!] ZERO articles with citation data found.")
        else:
            print("[+] Valid citation strings found:")
            for row in valid_rows:
                print(f"  PMID {row[0]}: {repr(row[1])[:80]}...")

            print("\n--- TEST 3: Verifying Pandas Math ---")
            df = pd.DataFrame(valid_rows, columns=['pmid', 'cited_by'])
            cited_by_str = df['cited_by'].astype(str).str.strip()
            df['Incoming_Citations_Smoothed'] = cited_by_str.str.count(';') + 1
            print(df.to_string(index=False))

    except Exception as e:
        print(f"[!] SQLite Error: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    run_citation_forensics()

In [ ]:
import os
import sqlite3
import time
import pandas as pd
import numpy as np

# --- 1. MOUNT GOOGLE DRIVE ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

# --- 2. DEFINE PATHS ---
PROCESSED_DIR = "/content/drive/MyDrive/Mesh-Network-Analysis-Main-Library/data/processed"
RELEVANCE_DB = os.path.join(PROCESSED_DIR, "DAC_Mesh_mean_relevancy.db")
CLEANED_DB = os.path.join(PROCESSED_DIR, "DAC_Mesh_cleaned_pmids.db")

def run_split_query_test(limit=500):
    print(f"<<< INITIATING SPLIT-QUERY TEST (LIMIT: {limit}) >>>")

    if not os.path.exists(RELEVANCE_DB) or not os.path.exists(CLEANED_DB):
        print(f"[!] Missing database files.")
        return

    start_time = time.time()

    # STEP 1: Extract Top Topological PMIDs
    print(f"\n[1] Extracting top {limit} PMIDs from Relevance DB...")
    conn_rel = sqlite3.connect(f"file:{RELEVANCE_DB}?mode=ro", uri=True)
    cursor_rel = conn_rel.cursor()
    cursor_rel.execute(f"SELECT pmid, score_betweenness_centrality FROM article_relevance_scores ORDER BY score_betweenness_centrality DESC LIMIT {limit}")
    rel_rows = cursor_rel.fetchall()
    conn_rel.close()

    if not rel_rows:
        print("[!] No records found in Relevance DB.")
        return

    # STEP 2: Format PMIDs (Strips any .0 float artifacts and converts to string)
    print("[2] Cleaning PMIDs for cross-reference...")
    clean_pmids = [str(r[0]).split('.')[0] for r in rel_rows]

    # Store base data for the merge
    rel_data_map = {clean_pmids[i]: rel_rows[i][1] for i in range(len(clean_pmids))}

    # STEP 3: Batch Lookup in Master DB
    print("[3] Querying Master DB for citation strings...")
    citation_map = {}
    conn_clean = sqlite3.connect(f"file:{CLEANED_DB}?mode=ro", uri=True)
    cursor_clean = conn_clean.cursor()

    # SQLite maximum variable limit is 999. Chunking at 900.
    chunk_size = 900
    for i in range(0, len(clean_pmids), chunk_size):
        chunk = clean_pmids[i:i + chunk_size]
        placeholders = ','.join('?' for _ in chunk)
        query = f"SELECT pmid, cited_by FROM pmids_table WHERE pmid IN ({placeholders})"
        cursor_clean.execute(query, chunk)

        for row in cursor_clean.fetchall():
            citation_map[str(row[0])] = row[1]

    conn_clean.close()

    # STEP 4: Merge and Calculate
    print("[4] Merging arrays and calculating citation counts...")
    merged_data = []
    for pmid in clean_pmids:
        cited_by_str = citation_map.get(pmid, None)
        merged_data.append({
            'pmid': pmid,
            'betweenness_centrality': rel_data_map[pmid],
            'cited_by_raw': cited_by_str
        })

    df = pd.DataFrame(merged_data)

    # Apply Pandas Math
    cited_by_series = df['cited_by_raw'].fillna('').astype(str).str.strip()
    is_empty = (cited_by_series == '') | (cited_by_series.str.lower() == 'nan') | (cited_by_series.str.lower() == 'none')
    df['incoming_citations'] = np.where(is_empty, 0, cited_by_series.str.count(';') + 1)

    exec_time = time.time() - start_time

    # STEP 5: Output Results
    print(f"\n--- TEST COMPLETE ---")
    print(f"Execution Time: {exec_time:.4f} seconds")
    print(f"Total PMIDs Queried: {len(df)}")

    matched_citations = (df['incoming_citations'] > 0).sum()
    print(f"Articles with >0 Citations: {matched_citations} out of {len(df)}")

    print("\n--- TOP 10 MERGED RESULTS ---")
    print(df[['pmid', 'betweenness_centrality', 'incoming_citations']].head(10).to_string(index=False))

if __name__ == "__main__":
    run_split_query_test(limit=500)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def analyze_reviewer_metrics(excel_filepath: str, output_image: str = 'Reviewer_Correlation_Analysis.png'):
    print(f"<<< Analyzing File: {excel_filepath} (First Sheet) >>>\n")

    # 1. Load Data from the first sheet of the Excel workbook
    try:
        df = pd.read_excel(excel_filepath, sheet_name=0, engine='openpyxl')
    except FileNotFoundError:
        print(f"[!] Error: Could not find file at {excel_filepath}")
        return
    except Exception as e:
        print(f"[!] Error reading Excel file: {e}")
        return

    # Filter out absolute zeros to prevent log-scale mathematical domain errors
    plot_df = df[(df['MRS_betweenness_centrality'] > 0) & (df['MRS_eigenvector_centrality'] > 0)].copy()

    # 2. Spearman Rank Correlation Calculation
    rho_bet, p_bet = spearmanr(plot_df['MRS_betweenness_centrality'], plot_df['article_count'])
    rho_eig, p_eig = spearmanr(plot_df['MRS_eigenvector_centrality'], plot_df['article_count'])

    print("--- SPEARMAN CORRELATION RESULTS ---")
    print(f"MRS_Betweenness vs Publication Count: Rho = {rho_bet:.4f} (p = {p_bet:.2e})")
    print(f"MRS_Eigenvector vs Publication Count: Rho = {rho_eig:.4f} (p = {p_eig:.2e})")

    # 3. Hierarchy Supremacy Evaluation (Top 10 Rankings)
    print("\n--- TOP 10 NODES: MRS_BETWEENNESS ---")
    top_bet = df.nlargest(10, 'MRS_betweenness_centrality')[['label', 'article_count', 'MRS_betweenness_centrality']]
    print(top_bet.to_string(index=False))

    print("\n--- TOP 10 NODES: MRS_EIGENVECTOR ---")
    top_eig = df.nlargest(10, 'MRS_eigenvector_centrality')[['label', 'article_count', 'MRS_eigenvector_centrality']]
    print(top_eig.to_string(index=False))

    # 4. Correlation Plot Generation
    print("\n<<< Generating Correlation Plots >>>")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # MRS Betweenness Plot
    sns.regplot(data=plot_df, x='article_count', y='MRS_betweenness_centrality', ax=axes[0],
                logx=True, scatter_kws={'alpha': 0.6, 'color': '#4c2b6d'}, line_kws={'color': 'red'})
    axes[0].set_xscale('log')
    axes[0].grid(True, which="both", ls="--", alpha=0.4)
    axes[0].set_title(f'MRS (Betweenness) vs Publication Volume\nSpearman Rho = {rho_bet:.3f}', fontsize=12, pad=10)
    axes[0].set_xlabel('Publication Count (Log Scale)', fontsize=11)
    axes[0].set_ylabel('MRS (Betweenness Centrality)', fontsize=11)

    # MRS Eigenvector Plot
    sns.regplot(data=plot_df, x='article_count', y='MRS_eigenvector_centrality', ax=axes[1],
                logx=True, scatter_kws={'alpha': 0.6, 'color': '#e8724d'}, line_kws={'color': 'red'})
    axes[1].set_xscale('log')
    axes[1].grid(True, which="both", ls="--", alpha=0.4)
    axes[1].set_title(f'MRS (Eigenvector) vs Publication Volume\nSpearman Rho = {rho_eig:.3f}', fontsize=12, pad=10)
    axes[1].set_xlabel('Publication Count (Log Scale)', fontsize=11)
    axes[1].set_ylabel('MRS (Eigenvector Centrality)', fontsize=11)

    plt.tight_layout()
    print(f"[+] Output successfully saved to: {output_image}")

if __name__ == "__main__":
    target_file = "results/DAC_Mesh_export.xlsx"
    analyze_reviewer_metrics(target_file)

In [ ]:
%cd /content/drive/MyDrive/Mesh-Network-Analysis-Main-Library

In [ ]:
!pip install biopython

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, ndcg_score

def calculate_k_metrics(df: pd.DataFrame, score_col: str, k_values: list) -> dict:
    sorted_df = df.sort_values(by=score_col, ascending=False)
    total_positives = sorted_df['y_true'].sum()
    metrics = {}
    for k in k_values:
        top_k = sorted_df.head(k)
        tp_k = top_k['y_true'].sum()
        recall_k = tp_k / total_positives if total_positives > 0 else 0
        precision_k = tp_k / k
        metrics[k] = {'recall': recall_k, 'precision': precision_k, 'tp': tp_k}
    return metrics

def get_distribution_stats(series: pd.Series) -> list:
    if series.empty:
        return ["0.000000"] * 5
    return [
        f"{series.mean():.6f}",
        f"{series.median():.6f}",
        f"{series.std():.6f}",
        f"{series.quantile(0.25):.6f}",
        f"{series.quantile(0.75):.6f}"
    ]

def run_pure_mechanistic_benchmark(resolved_csv_path: str, relevance_db_path: str):
    print("\n" + "="*80)
    print("<<< OECD AOP BENCHMARK: PURE MECHANISTIC EDGES VS. NAIVE QUERY >>>")
    print("="*80)

    # ---------------------------------------------------------
    # 1. LOAD DATASETS
    # ---------------------------------------------------------
    if not os.path.exists(resolved_csv_path):
        raise FileNotFoundError(f"Reference CSV not found: {resolved_csv_path}")
    ref_df = pd.read_csv(resolved_csv_path, sep=';')
    oecd_pmids = set(ref_df[ref_df['PMID'] != 'NOT_FOUND']['PMID'].astype(str).tolist())
    total_oecd = len(oecd_pmids)

    if not os.path.exists(relevance_db_path):
        raise FileNotFoundError(f"Database not found: {relevance_db_path}")

    print(f"Loading local network data from {os.path.basename(relevance_db_path)}...")
    conn = sqlite3.connect(f"file:{relevance_db_path}?mode=ro", uri=True)
    pipeline_df = pd.read_sql_query(
        "SELECT pmid, score_betweenness_centrality, score_eigenvector_centrality, contributing_seeds FROM article_relevance_scores",
        conn
    )
    conn.close()

    primary_node = "Dermatitis, Allergic Contact"
    pipeline_df['y_true'] = pipeline_df['pmid'].astype(str).isin(oecd_pmids).astype(int)
    pipeline_df['has_acd'] = pipeline_df['contributing_seeds'].str.contains(primary_node, na=False, regex=False).astype(int)

    # ---------------------------------------------------------
    # 2. CALCULATE AND APPLY NODE PENALTY (BEFORE EDGE FILTER)
    # ---------------------------------------------------------
    # Extract the pure weight of the primary node from 'dangling' N=1 articles
    acd_only_df = pipeline_df[pipeline_df['contributing_seeds'].str.strip() == primary_node]
    node_penalty_bet = acd_only_df['score_betweenness_centrality'].median() if not acd_only_df.empty else 0.0
    node_penalty_eig = acd_only_df['score_eigenvector_centrality'].median() if not acd_only_df.empty else 0.0

    # Subtract this weight from any article containing the primary node
    is_naive = pipeline_df['has_acd'] == 1

    pipeline_df['mech_bet'] = pipeline_df['score_betweenness_centrality']
    pipeline_df.loc[is_naive, 'mech_bet'] = (pipeline_df.loc[is_naive, 'score_betweenness_centrality'] - node_penalty_bet).clip(lower=0)

    pipeline_df['mech_eig'] = pipeline_df['score_eigenvector_centrality']
    pipeline_df.loc[is_naive, 'mech_eig'] = (pipeline_df.loc[is_naive, 'score_eigenvector_centrality'] - node_penalty_eig).clip(lower=0)

    print(f"\n--- 1. NOISE MITIGATION (PRIMARY HUB NEUTRALIZATION) ---")
    print(f"Target Subtraction Node: '{primary_node}'")
    print(f"Betweenness Hub Penalty: {node_penalty_bet:.6f}")
    print(f"Eigenvector Hub Penalty: {node_penalty_eig:.6f}")
    print("-> Penalty successfully subtracted from all matching naive and network articles.")

    # ---------------------------------------------------------
    # 3. APPLY EDGE-BRIDGING FILTER (N >= 2)
    # ---------------------------------------------------------
    pipeline_df['edge_bridging'] = (pipeline_df['contributing_seeds'].str.count(';') >= 1).astype(int)
    filtered_net = pipeline_df[pipeline_df['edge_bridging'] == 1].copy()
    filtered_naive = filtered_net[filtered_net['has_acd'] == 1].copy()

    print(f"\n--- 2. NOISE MITIGATION (EDGE FILTERING) ---")
    print(f"Removed {len(pipeline_df) - len(filtered_net):,} 'dangling node' (N=1) articles.")
    print(f"Retained {len(filtered_net):,} articles actively bridging mechanistic edges.")

    # ---------------------------------------------------------
    # 4. OVERALL RECALL & TOPOLOGY EXCLUSIVE HITS
    # ---------------------------------------------------------
    network_pmids = set(filtered_net['pmid'].astype(str))
    naive_pmids = set(filtered_naive['pmid'].astype(str))

    oecd_in_naive = oecd_pmids.intersection(naive_pmids)
    oecd_in_network = oecd_pmids.intersection(network_pmids)
    topology_exclusive = oecd_in_network - naive_pmids

    recall_data = {
        "Metric": ["Filtered Corpus Volume", "OECD Target Set", "Captured OECD Papers", "Absolute Recall"],
        "Filtered Naive (ACD + Edge)": [f"{len(naive_pmids):,}", total_oecd, len(oecd_in_naive), f"{(len(oecd_in_naive)/total_oecd)*100:.1f}%"],
        "Filtered Expanded Network": [f"{len(filtered_net):,}", total_oecd, len(oecd_in_network), f"{(len(oecd_in_network)/total_oecd)*100:.1f}%"]
    }
    print(f"\n--- 3. OPERATIONAL RECALL ON MECHANISTIC EDGES ---")
    print(pd.DataFrame(recall_data).to_string(index=False))
    print(f"\n> Topology-Exclusive: {len(topology_exclusive)} OECD papers found by the network lack the primary disease node entirely.")

    # ---------------------------------------------------------
    # 5. DISTRIBUTIONAL COMPARISON (PURE MECHANISTIC SCORES)
    # ---------------------------------------------------------
    print(f"\n--- 4. PURE MECHANISTIC DISTRIBUTIONAL COMPARISON ---")
    print("(Central disease hub weight removed; isolates secondary edge strengths)")

    mech_bet_naive = get_distribution_stats(filtered_naive['mech_bet'])
    mech_bet_net = get_distribution_stats(filtered_net['mech_bet'])

    dist_data_bet = {
        "Dataset (Mechanistic Betweenness)": [f"Naive Subset (N={len(filtered_naive):,})", f"Expanded Network (N={len(filtered_net):,})"],
        "Mean": [mech_bet_naive[0], mech_bet_net[0]],
        "Median": [mech_bet_naive[1], mech_bet_net[1]],
        "Std Dev": [mech_bet_naive[2], mech_bet_net[2]],
        "IQR (25th - 75th)": [f"{mech_bet_naive[3]} - {mech_bet_naive[4]}", f"{mech_bet_net[3]} - {mech_bet_net[4]}"]
    }
    print("\nA. Mechanistic Betweenness ARS Spread:")
    print(pd.DataFrame(dist_data_bet).to_string(index=False))

    mech_eig_naive = get_distribution_stats(filtered_naive['mech_eig'])
    mech_eig_net = get_distribution_stats(filtered_net['mech_eig'])

    dist_data_eig = {
        "Dataset (Mechanistic Eigenvector)": [f"Naive Subset (N={len(filtered_naive):,})", f"Expanded Network (N={len(filtered_net):,})"],
        "Mean": [mech_eig_naive[0], mech_eig_net[0]],
        "Median": [mech_eig_naive[1], mech_eig_net[1]],
        "Std Dev": [mech_eig_naive[2], mech_eig_net[2]],
        "IQR (25th - 75th)": [f"{mech_eig_naive[3]} - {mech_eig_naive[4]}", f"{mech_eig_net[3]} - {mech_eig_net[4]}"]
    }
    print("\nB. Mechanistic Eigenvector ARS Spread:")
    print(pd.DataFrame(dist_data_eig).to_string(index=False))

    # ---------------------------------------------------------
    # 6. RANKING PERFORMANCE COMPARISON
    # ---------------------------------------------------------
    print(f"\n--- 5. RANKING PERFORMANCE COMPARISON (Mechanistic Betweenness) ---")

    def get_comparison_metrics(df, score_col):
        if df['y_true'].sum() == 0 or len(df[df['y_true']==0]) == 0:
            return [0, 0, 0]
        roc = roc_auc_score(df['y_true'], df[score_col])
        stat, p_val = mannwhitneyu(df[df['y_true']==1][score_col], df[df['y_true']==0][score_col], alternative='greater')
        ndcg = ndcg_score(np.asarray([df['y_true'].values]), np.asarray([df[score_col].values]), k=1000)
        return [f"{roc:.4f}", f"{p_val:.2e}", f"{ndcg:.4f}"]

    naive_metrics = get_comparison_metrics(filtered_naive, 'mech_bet')
    net_metrics = get_comparison_metrics(filtered_net, 'mech_bet')

    rank_data = {
        "Metric": ["ROC-AUC (Signal Isolation)", "Mann-Whitney U p-value", "NDCG@1000"],
        "Performance in Filtered Naive": naive_metrics,
        "Performance in Filtered Network": net_metrics
    }
    print(pd.DataFrame(rank_data).to_string(index=False))

    # ---------------------------------------------------------
    # 7. COMPREHENSIVE METRICS (FILTERED NETWORK)
    # ---------------------------------------------------------
    print(f"\n--- 6. FULL COMPREHENSIVE METRICS (Edge-Filtered Network Only) ---")

    for score_col, label in [('mech_bet', 'Mechanistic Betweenness'), ('mech_eig', 'Mechanistic Eigenvector')]:
        print(f"\n{'='*50}\n  ANALYSIS: {label}\n{'='*50}")

        y_true = filtered_net['y_true']
        y_score = filtered_net[score_col]

        roc_auc = roc_auc_score(y_true, y_score)
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        pr_auc = auc(recall, precision)

        print(f"\n[A] Binary Classification")
        print(f"    - ROC-AUC: {roc_auc:.4f}")
        print(f"    - PR-AUC:  {pr_auc:.4f}")

        print(f"\n[B] Information Retrieval (Top-K & NDCG)")
        y_true_arr = np.asarray([y_true.values])
        y_score_arr = np.asarray([y_score.values])

        for k in [1000, 10000, 50000]:
            print(f"    - NDCG@{k:<6}: {ndcg_score(y_true_arr, y_score_arr, k=k):.4f}")

        k_cutoffs = [100, 500, 1000, 5000]
        k_metrics = calculate_k_metrics(filtered_net, score_col, k_cutoffs)
        for k in k_cutoffs:
            m = k_metrics[k]
            print(f"    - @K={k:<5} | Recall: {m['recall']*100:>5.2f}% | Precision: {m['precision']*100:>5.2f}% | Found: {m['tp']}")

    print("\n" + "="*80)
    print(f"[+] Operational Output Complete. No files written to disk.")
    print("="*80 + "\n")

if __name__ == "__main__":
    run_pure_mechanistic_benchmark(
        resolved_csv_path="data/reference_raw/oecd_resolved_citations.csv",
        relevance_db_path="data/processed/DAC_Mesh_mean_relevancy.db"
    )

In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def probe_endpoint(name: str, url: str):
    print(f"\nProbing: {name}")
    print(f"URL: {url}")

    try:
        response = requests.get(url, verify=False, timeout=10)
        print(f"Status: {response.status_code}")

        if response.status_code == 200:
            if "application/json" in response.headers.get("Content-Type", ""):
                print("Result: SUCCESS (JSON API Data Returned)")
            else:
                print("Result: SUCCESS (But returned HTML/Text, not JSON)")
        elif response.status_code == 404:
            print("Result: 404 NOT FOUND (Path does not exist)")
        elif response.status_code >= 500:
            print(f"Result: {response.status_code} SERVER ERROR (Backend is down)")
        else:
            print(f"Result: Unexpected status {response.status_code}")

    except Exception as e:
        print(f"Result: CONNECTION FAILED - {e}")

if __name__ == "__main__":
    print("<<< INITIATING NIH ICITE INFRASTRUCTURE PROBE >>>")

    # 1. Is the main website even online?
    probe_endpoint("Base Web Interface", "https://icite.od.nih.gov/")

    # 2. Did they disable the bulk query parameter? (Testing the single-item URL path)
    probe_endpoint("Single-Item Path API", "https://icite.od.nih.gov/api/pubs/23456789")

    # 3. Did they migrate to a versioned URL? (v1)
    probe_endpoint("Versioned API (v1)", "https://icite.od.nih.gov/api/v1/pubs/23456789")

    # 4. Did they migrate to a versioned URL? (v2)
    probe_endpoint("Versioned API (v2)", "https://icite.od.nih.gov/api/v2/pubs/23456789")

    # 5. Did they drop the 'api' subdirectory?
    probe_endpoint("Root Pubs Endpoint", "https://icite.od.nih.gov/pubs/23456789")

In [ ]:
# Download the latest iCite Database Snapshot (Metadata)
!wget -O icite_metadata.zip https://nih.figshare.com/ndownloader/files/45532557

In [ ]:
import os
import sqlite3
import time
import pandas as pd

ZIP_PATH = "icite_metadata.zip"
DB_PATH = "master_citation_graph.db"
CHUNKSIZE = 250000 # Optimal balance for Colab RAM

def build_production_citation_db(zip_file: str, db_file: str):
    print(f"\n<<< INITIATING STREAMING BUILD: {zip_file} -> {db_file} >>>")

    if os.path.exists(db_file):
        os.remove(db_file)

    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    # 1. Apply aggressive SQLite pragmas for massive bulk inserts
    cursor.execute("PRAGMA journal_mode = OFF;")
    cursor.execute("PRAGMA synchronous = 0;")
    cursor.execute("PRAGMA cache_size = 1000000;")
    cursor.execute("PRAGMA locking_mode = EXCLUSIVE;")

    cursor.execute('''
        CREATE TABLE occ_citations (
            pmid INTEGER PRIMARY KEY,
            cited_by TEXT,
            references_list TEXT
        )
    ''')
    conn.commit()

    print("Streaming data in chunks. This may take 15-30 minutes depending on Colab I/O...")
    start_time = time.time()
    rows_inserted = 0

    try:
        # 2. Stream directly from the ZIP to avoid 50GB disk bloat
        # Note: iCite columns are typically 'pmid', 'cited_by', 'references'.
        # If the schema changes, update usecols accordingly.
        csv_stream = pd.read_csv(
            zip_file,
            compression='zip',
            chunksize=CHUNKSIZE,
            usecols=['pmid', 'cited_by', 'references'],
            dtype=str # Read all as strings to prevent NaN casting issues
        )

        for chunk in csv_stream:
            chunk = chunk.fillna("")
            chunk_data = list(chunk.itertuples(index=False, name=None))

            cursor.executemany('''
                INSERT OR IGNORE INTO occ_citations (pmid, cited_by, references_list)
                VALUES (?, ?, ?)
            ''', chunk_data)

            rows_inserted += len(chunk)
            print(f"  -> Inserted {rows_inserted:,} rows...")

    except ValueError as e:
        print(f"\n[!] CSV Schema Mismatch. The NIH may have changed column headers: {e}")
        print("Please check the CSV headers in the latest iCite release.")
        return

    # 3. Build indices AFTER insertion for maximum speed
    print("\nBuilding database indices for fast graph traversal (This takes a moment)...")
    cursor.execute("CREATE INDEX idx_cited_by ON occ_citations(cited_by)")
    conn.commit()
    conn.close()

    elapsed = time.time() - start_time
    print(f"\n[+] Production Database Built! {rows_inserted:,} rows processed in {elapsed/60:.2f} minutes.")

if __name__ == "__main__":
    if not os.path.exists(ZIP_PATH):
        print(f"Error: {ZIP_PATH} not found. Did you run the wget command?")
    else:
        build_production_citation_db(ZIP_PATH, DB_PATH)

In [ ]:
import sqlite3
import time

DB_PATH = "master_citation_graph.db"
BATCH_SIZE = 999

# We are testing the exact PMIDs from your failed API logs
TEST_PMIDS = [
    15795, 123700, 124011, # Old PMIDs
    38864900, 38884533, 38885151 # New PMIDs
]

def execute_production_expansion(db_file: str, initial_pmids: list, generations: int):
    print(f"\n<<< EXECUTING {generations}-GENERATION EXPANSION >>>")
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    generation_map = {int(pmid): "P0" for pmid in initial_pmids}
    current_parents = set(int(pmid) for pmid in initial_pmids)

    start_time = time.time()

    for gen in range(1, generations + 1):
        gen_label = f"G{gen}"
        print(f"\n> Expanding {len(current_parents):,} parents to find {gen_label} children...")

        next_generation_candidates = set()
        parent_list = list(current_parents)

        for i in range(0, len(parent_list), BATCH_SIZE):
            batch = parent_list[i:i + BATCH_SIZE]
            placeholders = ",".join("?" for _ in batch)

            cursor.execute(f'''
                SELECT cited_by, references_list
                FROM occ_citations
                WHERE pmid IN ({placeholders})
            ''', batch)

            for row in cursor.fetchall():
                cited_by_str = row[0]
                refs_str = row[1]

                combined = f"{cited_by_str};{refs_str}"
                # Split, strip, and cast to int, ignoring empty strings
                children = [int(p.strip()) for p in combined.split(";") if p.strip().isdigit()]
                next_generation_candidates.update(children)

        # Identify strictly novel children
        new_children = next_generation_candidates - set(generation_map.keys())

        for child in new_children:
            generation_map[child] = gen_label

        print(f"  -> Found {len(new_children):,} novel {gen_label} children.")
        current_parents = new_children

        if not current_parents:
            print("  -> No new children found. Halting early.")
            break

    conn.close()
    elapsed = time.time() - start_time

    print(f"\n[+] Expansion Complete in {elapsed:.4f} seconds.")
    print(f"Total Unique PMIDs in Local Network: {len(generation_map):,}")

    dist = {}
    for pmid, gen in generation_map.items():
        dist[gen] = dist.get(gen, 0) + 1

    for gen, count in sorted(dist.items()):
        print(f"  {gen}: {count:,} papers")

if __name__ == "__main__":
    execute_production_expansion(DB_PATH, TEST_PMIDS, generations=2)